# Assignment: Video Quality Inference

To this point in the class, you have learned various techniques for leading and analyzing packet captures of various types, generating features from those packet captures, and training and evaluating models using those features.

In this assignment, you will put all of this together, using a network traffic trace to train a model to automatically infer video quality of experience from a labeled traffic trace.

## Getting the data

Everything you need is in the `data/` folder next to this notebook (originally from the course data repository on GitHub, [noise-courses/data](https://github.com/noise-courses/data), under `video-qoe/`). The files are:

| File | Used in | Size |
|---|---|---|
| `data/netflix.pcap` | Part 1 | 22 MB |
| `data/netflix_dataset.pkl.xz` *or* `data/video_dataset.pkl.xz` | Part 2 | 22 MB / 61 MB compressed |
| `data/netflix_session.pkl` | Part 3 | 130 KB |

## Part 1: Warmup

The first part of this assignment builds directly on the hands-on activities but extends them slightly.

### Extract Features from the Network Traffic

Load the `data/netflix.pcap` file, which is a packet trace that includes network traffic.



In [4]:
from netml.pparser.parser import PCAP
import pandas as pd
netflix_pcap = PCAP('data/netflix.pcap', flow_ptks_thres=2, verbose=10)
netflix_pcap.pcap2pandas()
netflix_pcap_df = netflix_pcap.df
netflix_pcap_df.head()

netflix_pcap = PCAP('data/netflix.pcap', flow_ptks_thres=2, verbose=10)
netflix_pcap.pcap2flows(interval=10)

pcap_file: data/netflix.pcap
ith_packets: 0
ith_packets: 10000
ith_packets: 20000
ith_packets: 30000
ith_packets: 40000
ith_packets: 50000
ith_packets: 60000
ith_packets: 70000
ith_packets: 80000
ith_packets: 90000
ith_packets: 100000
ith_packets: 110000
ith_packets: 120000
ith_packets: 130000
ith_packets: 140000
len(flows): 275
total number of flows: 275. Num of flows < 2 pkts: 91, and >=2 pkts: 184 without timeout splitting.
kept flows: 184. Each of them has at least 2 pkts after timeout splitting.
0th_flow: len(pkts): 12
After splitting flows, the number of subflows: 247 and each of them has at least 2 packets.


### Identifying the Service Type

Use the DNS traffic to filter the packet trace for Netflix traffic.

In [9]:
netflix_pcap.q_interval = 0.9 # Was getting an error realted to q_interval. Asked claude for help in order to debug
netflix_pcap.flow2features('STATS', fft=False, header=False)

dns_traffic = netflix_pcap_df[netflix_pcap_df['is_dns']]
dns_traffic = dns_traffic[dns_traffic['dns_resp'].notna()].copy()

dns_traffic['dns_query'] = dns_traffic['dns_query'].str[0]
dns_traffic['dns_resp'] = dns_traffic['dns_resp'].str[0]

# Now filtering for netflix in order to get the ips for netflix
dns_traffic = dns_traffic[dns_traffic['dns_query'].str.contains('netflix|nflx', case=False, na=False)]
display(dns_traffic[['dns_query', 'dns_resp']])

#Print the ips
netflix_ips = set(dns_traffic['dns_resp'])
print(len(netflix_ips), 'Netflix IPs:', netflix_ips)

True


,dns_query,dns_resp
101,ipv4-c001-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.130
102,www.eu-west-1.prodaa.netflix.com.,52.19.39.146
103,customerevents.eu-west-1.prodaa.netflix.com.,52.210.19.176
104,ipv4-c024-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.153
219,occ-0-56-55.1.nflxso.net.,198.38.120.137
228,ipv4-c072-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.167
1019,push.prod.eu-west-1.prodaa.netflix.com.,34.252.77.54
1566,ipv4-c005-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.134
1598,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.164
1600,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net.,198.38.120.166


15 Netflix IPs: {'52.48.148.78', '34.252.77.54', '52.48.8.150', '198.38.120.130', '198.38.120.134', '52.210.19.176', '52.19.39.146', '198.38.120.164', '198.38.120.166', '52.210.133.255', '198.38.120.153', '52.208.128.101', '198.38.120.167', '198.38.120.137', '198.38.120.162'}


### Generate Statistics

Generate statistics and features for the Netflix traffic flows. Use the `netml` library or any other technique that you choose to generate a set of features that you think would be good features for your model. 

In [14]:
# Now we will generate statistics based on the ips of the netflix stuff from aboce. 
cols = ['duration','pkts_rate','bytes_rate','mean','std','q1','median','q3','min','max','num_pkts','num_bytes']
features = ['duration', 'pkts_rate', 'bytes_rate', 'mean', 'std', 'num_pkts', 'num_bytes']
stats = pd.DataFrame(netflix_pcap.features, columns=cols)
stats['src'] = [f[0] for f in netflix_pcap.fids]
stats['dst'] = [f[1] for f in netflix_pcap.fids]

stats = stats[stats['src'].isin(netflix_ips) | stats['dst'].isin(netflix_ips)]
stats.reset_index()

netflix_features = stats[['src', 'dst'] + features]
netflix_features.shape

(151, 9)

**Write a brief justification for the features that you have chosen.**

**Feature justification**

| Feature | Justification |
|---|---|
| `bytes_rate` | Bytres per second is probably the most important feature to understand video quality. Higher byte reate means larger video and high resolution vidoes. |
| `pkts_rate` | Shows how busy the flow is, and can show if the video is actively downloading if high byte and packet rate or control flow is high pkt rate and low byte.  |
| `mean` (packet size) | Video traffic mixes full-size data packets (~1500 B) with small ACKs. Need mean to see how much video is mixed in. |
| `std` (packet size) | A high std means a mix of large video packets and small control packets, which signals bursty segment downloads rather than steady traffic. |
| `duration` |  Within a 10 s window it shows whether the flow downloaded continuously or only in a short burst. |
| `num_pkts` | The total packet count in the window. Unlike `pkts_rate`, it isn't divided by duration, so it captures the total amount of activity even when a flow is only active for part of the window. |
| `num_bytes` | The total bytes in the window. It approximates how much video was downloaded in each 10 s interval which tracks the segment sizes that dominate quality prediction. |

### Inferring Segment downloads

In addition to the features that you could generate using the `netml` library or similar, add to your feature vector a "segment downloads rate" feature, which indicates the number of video segments downloaded for a given time window.

Note: If you are using the `netml` library, generating features with `SAMP` style options may be useful, as this option gives you time windows, and you can then simply add the segment download rate to that existing dataframe.

## Part 2: Video Quality Inference

You will now load the complete video dataset from a previous study to train and test models based on these features to automatically infer the quality of a streaming video flow.

For this part of the assignment, you will need two pickle files, both in the `data/` folder:

- `data/netflix_session.pkl` (used in Part 3)
- `data/video_dataset.pkl.xz`

The dataset is stored xz-compressed (61 MB compressed, about 265 MB uncompressed). You do not need to decompress it first: `pd.read_pickle` reads `.pkl.xz` files directly, e.g. `pd.read_pickle('data/video_dataset.pkl.xz')`. (The uncompressed `.pkl` is git-ignored so it never gets committed.)

A smaller Netflix-only dataset (`data/netflix_dataset.pkl.xz`, 52,279 samples with 251 features) is available in the same folder and may be used instead of the multi-service `video_dataset.pkl.xz` (204,713 samples with 170 features).


### Load the File

Load the video dataset pickle file.

### Clean the File

1. The dataset contains video resolutions that are not valid. Remove entries in the dataset that do not contain a valid video resolution. Valid resolutions are 240, 360, 480, 720, 1080.

2. The file also contains columns that are unnecessary (in fact, unhelpful!) for performing predictions. Identify those columns, and remove them.

**Briefly explain why you removed those columns.**

### Prepare Your Data

Prepare your data matrix, determine your features and labels, and perform a train-test split on your data.

### Train and Tune Your Model

1. Select a model of your choice.
2. Train the model using your training data.

### Tune Your Model

Perform hyperparameter tuning to find optimal parameters for your model.

### Evaluate Your Model

Evaluate your model accuracy according to the following metrics:

1. Accuracy
2. F1 Score
3. Confusion Matrix
4. ROC/AUC

## Part 3: Predict the Ongoing Resolution of a Real Netflix Session

Now that you have your model, it's time to put it in practice!

Use a preprocessed Netflix video session to infer **and plot** the resolution at 10-second time intervals. The session is `netflix_session.pkl` (in the `data/` folder; 59 ten-second windows with the same 251 features as `netflix_dataset.pkl`, plus the true `resolution` so you can check yourself):

```python
session = pd.read_pickle('data/netflix_session.pkl')
```